# Public Sentiment and Discussion Patterns Surrounding Formula 1 Safety on X

**Course:** COSC2671 Social Media and Network Analytics  
**Assignment:** Assignment 2  
**Topic:** Formula 1 safety discussions on Youtube 

This notebook investigates how users on Youtube discuss Formula 1 safety issues. The analysis combines text preprocessing, exploratory data analysis, sentiment analysis, topic modelling, and network analysis to understand the main themes and patterns in public discussion.

## 1. Introduction (draft 1)

### Discussion
 This study analyses public discussion on Youtube around Formula 1 safety to identify dominant themes, sentiment patterns, and network structures within safety-related conversations.

Possible research questions:

1. What are the most common topics in F1 safety discussions on Youtube?
2. Is public sentiment mostly positive, negative, or neutral?
3. Which hashtags, keywords, or user mentions are most central in the discussion network?
4. How do sentiment and topic patterns connect with network communities?

Social media platforms have become a primary source of discussion, opinion formation, and exchange of information. Users not only express their views through text but also interact with one another, forming complex networks of influence and engagement. 
As a result, analysing online content requires more than just traditional text mining techniques — it demands an integrated approach that considers both what people say and how they are connected.

This study explores discussions surrounding *how does social media discussion influence perceptions of safety in Formula 1 after major accidents* by combining Natural Language Processing (NLP) and network analysis. 
The goal is to move beyond surface-level insights and uncover deeper patterns in sentiment, and interaction structures, by leveraging methods such as sentiment analysis, topic modelling, and transformer-based language models. The textual content of user-generated data can be systematically examined. At the same time, constructing and analysing interaction networks enables the identification of influential entities, community structures, and patterns of information flow.

## 2. Setup and Imports

Install the required libraries first if needed:

In [1]:
!pip install google-api-python-client pandas numpy matplotlib seaborn nltk textblob langdetect scikit-learn networkx python-louvain wordcloud transformers torch sentence-transformers deep_translator

In [2]:
# Core data tools
import os
import re
import json
import time
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# YouTube API
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

# Language detection
from langdetect import detect, LangDetectException

# NLP
import nltk
from nltk.tokenize import TweetTokenizer
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.sentiment import SentimentIntensityAnalyzer
from textblob import TextBlob

# Feature extraction and topic modelling
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation, TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity

# Network analysis
import networkx as nx

# Optional community detection
try:
    import community as community_louvain
    HAS_LOUVAIN = True
except ImportError:
    HAS_LOUVAIN = False

# Optional BERT / sentence embeddings
try:
    from sentence_transformers import SentenceTransformer
    HAS_SENTENCE_TRANSFORMERS = True
except ImportError:
    HAS_SENTENCE_TRANSFORMERS = False

# Download NLTK resources if needed
nltk.download('stopwords')
nltk.download('vader_lexicon')
nltk.download('wordnet')
nltk.download('omw-1.4')

pd.set_option('display.max_colwidth', 120)
sns.set_theme(style='whitegrid')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\davin\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\davin\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\davin\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\davin\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


## 3. YouTube API

In [3]:
# ── YouTubeProcessing.py ───────────────────────────────────
# Provides the YouTubeProcessing class with a .process() method.
# Centralises tokenisation + stopword filtering (no stemming).
from YouTubeProcessing import YouTubeProcessing

# ── youtubeTextProcessing.py ───────────────────────────────
# Provides processText() which adds Porter stemming on top.
# This function is what youtubeTextProcessing.py uses in its main loop.


print('Helper modules imported.')
print('  YouTubeProcessing.process() — tokenise + filter (no stemming)')
print('  processText()               — tokenise + stem + stopword removal')

Helper modules imported.
  YouTubeProcessing.process() — tokenise + filter (no stemming)
  processText()               — tokenise + stem + stopword removal


## 5. Collect Video Metadata

This function searches YouTube for relevant videos and stores key metadata. Video-level metadata is useful for EDA and for linking comment trends back to popular videos.

In [ ]:
from fetchYoutubeAPI import (
    SEARCH_QUERIES,
    search_videos_for_query,
    get_video_details
)

from youtubeClient import youtubeClient

client = youtubeClient()

MAX_TOTAL_VIDEOS = 60
VIDEOS_PER_QUERY = 10

all_items = []

for query in SEARCH_QUERIES:

    # Stop once enough videos collected
    if len(all_items) >= MAX_TOTAL_VIDEOS:
        break

    try:
        print(f"Searching: {query}")

        items = search_videos_for_query(
            client=client,
            query=query,
            max_videos=VIDEOS_PER_QUERY,
            published_after="2020-01-01T00:00:00Z"
        )

        all_items.extend(items)

        time.sleep(0.2)

    except HttpError as e:
        print(f"API error for query '{query}': {e}")

# Convert raw search results into video IDs
video_ids = [
    item["id"]["videoId"]
    for item in all_items
    if "id" in item and "videoId" in item["id"]
]

# Remove duplicates
video_ids = list(dict.fromkeys(video_ids))

# Final cap on total videos
video_ids = video_ids[:MAX_TOTAL_VIDEOS]

# Fetch detailed metadata/statistics
videos = get_video_details(client, video_ids)

video_df = pd.DataFrame(videos)

print(video_df.shape)
video_df.head()

Searching: Formula 1 crash analysis
Searching: F1 major accidents analysis
Searching: Formula 1 safety discussion
Searching: F1 halo safety analysis
Searching: Formula 1 dangerous crashes
Searching: Jules Bianchi crash analysis
Searching: Romain Grosjean Bahrain crash analysis
Searching: F1 crash reaction
Searching: Formula 1 driver safety
Searching: F1 FIA safety regulations
Searching: Formula 1 accidents documentary
Searching: F1 safety improvements
Searching: Formula 1 controversial crashes
Searching: F1 race accidents analysis
Searching: Formula 1 crash aftermath discussion
(492, 10)


,title,videoId,channelTitle,publishedAt,viewCount,likeCount,commentCount,durationSecs,category,comments
0,The Race Start Chaos & Major Incidents from The Miami Grand Prix | Jolyon Palmer's F1 TV Analysis,4703BVI4Giw,FORMULA 1,2026-05-05T14:30:35Z,106003,2832,315,849,safety_discussion,[]
1,The Horrifying Tragedy F1 Wants You To Forget..,6D4j1P0RcL4,DailyFuelUp,2025-04-11T12:00:15Z,766143,7223,565,696,safety_discussion,[]
2,Grosjean's Insane Fireball Crash | Formula 1: Drive To Survive S3 | Netflix,7YMjw2sjXqU,Still Watching Netflix,2021-03-23T12:00:00Z,47852599,714695,21314,409,safety_discussion,[]
3,3D Reconstruction of Gilles Villeneuve’s F1 Fatal Crash.,7O-PHymGHxI,3DCrash,2026-02-24T19:10:27Z,188892,2895,241,506,safety_discussion,[]
4,How Grosjean's Accident Happened | 2020 Bahrain Grand Prix | Jolyon Palmer Analysis,EyTeDaiUL6s,FORMULA 1,2020-12-01T19:30:01Z,2864441,56258,2311,692,safety_discussion,[]


In [5]:
# # Save raw video metadata
# os.makedirs("data", exist_ok=True)
# video_df.to_csv("data/f1_safety_videos.csv", index=False)
# video_df.to_json("data/f1_safety_videos.json", orient="records", indent=2)
# print("Saved video metadata.")

## 6. Collect YouTube Comments

The function below collects top-level comments from each video. Replies can be added later, but top-level comments are usually enough for sentiment, topic modelling, and co-occurrence networks.

The notebook saves raw comments so the API does not need to be called repeatedly.

In [6]:
from fetchYoutubeAPI import get_comments_for_video
from youtubeClient import youtubeClient

client = youtubeClient()

MAX_COMMENTS_PER_VIDEO = 500

# Collect comments
all_comments = []

for idx, row in video_df.iterrows():

    print(
        f"Collecting comments for "
        f"{idx+1}/{len(video_df)}: "
        f"{row['title'][:80]}"
    )

    comments = get_comments_for_video(
        client=client,
        video_id=row["videoId"],
        max_comments=MAX_COMMENTS_PER_VIDEO
    )

    # Add video metadata into each comment
    for c in comments:
        c["videoId"] = row["videoId"]
        c["videoTitle"] = row["title"]

    all_comments.extend(comments)

comments_df = pd.DataFrame(all_comments)

print(comments_df.shape)
comments_df.head()

KeyboardInterrupt: 

In [ ]:
# # Save raw comments
# comments_df.to_csv("data/f1_safety_comments_raw.csv", index=False)
# comments_df.to_json("data/f1_safety_comments_raw.json", orient="records", indent=2)
# print("Saved raw comments.")

## 7. Load Saved Data

After collecting data once, use this section in later runs to avoid spending API quota.

In [ ]:
# Uncomment these lines if you already collected and saved the data
# video_df = pd.read_csv("data/f1_safety_videos.csv")
# comments_df = pd.read_csv("data/f1_safety_comments_raw.csv")

print(video_df.shape)
print(comments_df.shape)